Project: "Music Streaming Habits Analyzer"
This brings everything together. Students reshape and combine multiple cleaned datasets to uncover business insights.
The Scenario: You are given three separate datasets: "User Demographics", "Track Metadata" (artist, genre, duration), and "Listening History" (who listened to what, and when).
* Combining Datasets: You must use pd.concat() to stitch together listening history from Q1 and Q2. Then, use pd.merge() to join the 'Listening History' table with the 'Track Metadata' table so you know the genre of the song played, acting like a SQL left join.
* Modifying DataFrames: You create a new calculated column called Minutes_Played by dividing the Milliseconds column by 60,000. Use .drop() to clear redundant system ID columns and .rename() poorly named columns.
* Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the time of day into "Morning", "Afternoon", "Evening" and "Night" based on the timestamp.
* Grouping and Aggregation: Using .groupby(), you group the data by 'Genre' and use .agg() to find the total minutes played, the average song duration, and the unique count of listeners for each genre.
* Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a heat-map-ready matrix showing "User Age Group" as rows, "Music Genre" as columns, and "Total Listens" as the values.

In [ ]:
# using to see  { Q1 }

import pandas as pd

dfQ1 = pd.read_csv('listening_history_q1.csv')

# print(dfQ1.columns)
dfQ1.shape 
dfQ1.index           # 0 -> 716 step 1

# -->>>>>>>>>  uisng to see NaT so the result is no NaT or Null
dropna_all = dfQ1.dropna(how='any')
dropna_all

# -->>>>>>>>>  using to see duplicated but no duplicated item
duplicates_q1 = dfQ1[dfQ1.duplicated(keep=False)]
duplicates_q1

data_clean = dfQ1.drop_duplicates(keep='first')
data_clean

In [ ]:
# using to see  { Q2 }

import pandas as pd

dfQ2 = pd.read_csv('listening_history_q2.csv')

print(dfQ2.columns)
dfQ2.index           # 0 -> 784 step 1

dropna_all = dfQ2.dropna(how='any')
dropna_all

dfplicated_q2 = dfQ2[dfQ2.duplicated(keep=False)]
dfplicated_q2

In [ ]:
import pandas as pd

track = pd.read_csv('track_metadata.csv')
track.index
track.shape      #  row 1-> 50 columns 5
track.head()

# using to see { track_metadata.csv } 
track_duplicated = track[track.duplicated(keep=False)]
track_duplicated                   

In [ ]:
import pandas as pd 

demographics = pd.read_csv('user_demographics.csv')
demographics
demographics.index       #  1 -> 200 step 1 columns 4
demographics.shape       

demographics_duplicated = demographics[demographics.duplicated(keep=False)]
demographics_duplicated

#### # Combining Datasets: You must use pd.concat() to stitch together listening history from Q1 and Q2. 
#### Then, use pd.merge() to join the 'Listening History' table with the 'Track Metadata'
#### table so you know the genre of the song played, acting like a SQL left join.

In [ ]:
import pandas as pd

full_history = pd.concat([dfQ1, dfQ2], axis=0)
full_history.index
full_history.shape          # start form 1 -> 1500  columns 4
full_history

# -->>>>>> uing this to change name in the columns 
full_history = full_history.rename(columns={'trk_ref_id': 'track_id'})

# -->>>>>> uisng this one to merge
final_df = pd.merge(full_history, track, on='track_id', how='left')
final_df

print(final_df.isnull().sum())

find_dupliacted = final_df[final_df.duplicated(keep=False)] 
find_dupliacted

final_df.to_excel('final_df.xlsx', index=False)

* Modifying DataFrames: You create a new calculated column called Minutes_Played by dividing the Milliseconds column by 60,000. Use .drop() to clear redundant system ID columns and .rename() poorly named columns.

In [153]:
import pandas as pd


if 'length_in_ms' in final_df.columns:
    # use to add what we have calculate in { final_df }
    final_df['minutes_played'] = final_df['length_in_ms'] / 60000
    # using drop to drop { length_in_ms }
    final_df = final_df.drop(columns=['length_in_ms'])


# using to break Date and Time in different columns 
final_df[['Date', 'Time']] = final_df['Listen_Date'].str.split(' ', expand=True)
# print(final_df[['Date', 'Time']].head())
final_df = final_df.drop(columns=['Listen_Date'])

final_df.rename(columns={
    'track_id': 'Track_ID',
    'artist_name': 'Artist',
    'timestamp': 'Listen_Date'
}, inplace=True)
    
final_df    
# final_df.shape

,listen_id,User_ID,Track_ID,Artist,Genre,internal_db_id,minutes_played,Date,Time
0,1,167,146,Artist_46,Electronic,119870,2.017700,2025-01-23,22:19:45
1,3,90,105,Artist_5,Electronic,138102,4.766383,2025-01-06,15:32:21
2,5,39,119,Artist_19,Hip-Hop,625830,2.239950,2025-02-12,11:12:04
3,6,126,149,Artist_49,Pop,334677,4.543617,2025-03-05,19:52:42
4,8,173,101,Artist_1,Pop,897606,3.433317,2025-01-04,05:56:14
...,...,...,...,...,...,...,...,...,...
1495,1494,175,126,Artist_26,Rock,319930,4.972533,2025-06-15,07:30:32
1496,1495,165,101,Artist_1,Pop,897606,3.433317,2025-05-22,13:18:12
1497,1496,49,128,Artist_28,Classical,617313,4.464183,2025-06-13,22:29:31
1498,1497,120,137,Artist_37,Hip-Hop,257504,4.323450,2025-04-26,02:20:18


* Applying Functions: You write a custom function (or use a lambda) and apply it via .apply() to categorize the time of day into "Morning", "Afternoon", "Evening" and "Night" based on the timestamp.

In [178]:
import pandas as pd

def categorize_time(timestamp):
    # using this to selection on the interger time before ' : ' 
    hour = int(timestamp.split(':')[0])
    
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

final_df['Time Period'] = final_df['Time'].apply(categorize_time)
final_df

# duplicates = final_df[final_df.duplicated(keep=False)]
# duplicates

,listen_id,User_ID,Track_ID,Artist,Genre,internal_db_id,minutes_played,Date,Time,Time Period
0,1,167,146,Artist_46,Electronic,119870,2.017700,2025-01-23,22:19:45,Night
1,3,90,105,Artist_5,Electronic,138102,4.766383,2025-01-06,15:32:21,Afternoon
2,5,39,119,Artist_19,Hip-Hop,625830,2.239950,2025-02-12,11:12:04,Morning
3,6,126,149,Artist_49,Pop,334677,4.543617,2025-03-05,19:52:42,Evening
4,8,173,101,Artist_1,Pop,897606,3.433317,2025-01-04,05:56:14,Morning
...,...,...,...,...,...,...,...,...,...,...
1495,1494,175,126,Artist_26,Rock,319930,4.972533,2025-06-15,07:30:32,Morning
1496,1495,165,101,Artist_1,Pop,897606,3.433317,2025-05-22,13:18:12,Afternoon
1497,1496,49,128,Artist_28,Classical,617313,4.464183,2025-06-13,22:29:31,Night
1498,1497,120,137,Artist_37,Hip-Hop,257504,4.323450,2025-04-26,02:20:18,Night


* Grouping and Aggregation: Using .groupby(), you group the data by 'Genre' and use .agg() to find the total minutes played, the average song duration, and the unique count of listeners for each genre.

In [173]:
import pandas as pd

summary = final_df.groupby('Genre').agg({
    'minutes_played': ['sum', 'mean'],
    'User_ID': 'nunique'
})
summary



minutes_played           User_ID
                      sum      mean nunique
Genre                                      
Classical      216.033783  4.500704      43
Electronic    1320.180167  3.636860     164
Hip-Hop       1089.167967  3.821642     154
Jazz           569.446450  3.409859     117
Pop           1069.306017  3.712868     162
Rock          1317.886817  3.776180     170

* Reshaping and Pivoting: Finally, you use pd.pivot_table() to create a heat-map-ready matrix showing "User Age Group" as rows, "Music Genre" as columns, and "Total Listens" as the values.

In [183]:
import pandas as pd

heatmap_matrix = pd.pivot_table(
    final_df, 
    values='listen_id', 
    index='Artist',    
    columns='Genre',   
    aggfunc='count'    
).fillna(0)            

heatmap_matrix = heatmap_matrix.astype(int)

print(heatmap_matrix.head())

Genre      Classical  Electronic  Hip-Hop  Jazz  Pop  Rock
Artist                                                    
Artist_0           0           0        0     0   25     0
Artist_1           0           0        0     0   34     0
Artist_10          0           0        0     0    0    28
Artist_11          0           0       28     0    0     0
Artist_12          0           0        0    28    0     0
